# Inhibitory Modulation Analysis

This notebook compares the original modular inhibitory modulation workflow with the newer `inhibitory_schur_modulation` analysis and merges the missing pieces. Numerical work remains in `inhibitory_modulation.py` so the notebook stays focused on setup, interpretation, and outputs.

## tl;dr

This is the main mechanistic fracture notebook. It compares `with_self` and `no_self` normalized matrix variants, then asks which source-to-receiver blocks, inhibitory feedback terms, motif edge sets, and explicit inhibitory/disinhibitory paths most shape the dominant dynamics. In the saved outputs, removing EE strongly lowers the dominant real eigenvalue, removing EI or IE raises it, and removing II has only a small direct effect. The Schur summaries show that inhibitory feedback brings the effective E operator back to the normalized full-system reference, while the Neumann model selects a longer I-I chain for `with_self` than for `no_self`.


## Comparison With The Newer Version

The earlier notebook already loaded a backbone matrix, built EE/EI/IE/II blocks, computed block statistics, ran simple ablation and scale perturbations, estimated an inhibitory feedback operator, selected E-I-I-...-E chain depth, and expanded the Neumann series.

The newer PDF adds several important pieces: E/I labels are recovered from the netlist when available; analyses are run side-by-side with self connections retained and removed; each variant is normalized to a chosen spectral radius; block perturbation reports both exact eigenvalue shifts and a first-order eigenvalue perturbation estimate; the Schur complement is evaluated at the dominant eigenvalue using `M_EE + M_EI inv(zI - M_II) M_IE`; resolvent responses to exciting all E or all I cells are ranked; motif enrichment and motif-removal stability effects are added; Neumann model selection includes a spectral-tail rule; top disinhibitory relays and explicit E-I-E / E-I-I-E paths are enumerated; and final tables are saved.

This merged notebook keeps the earlier clean function-outside-notebook design, but uses the newer perturbation analysis and adds the missing sections listed above.

## Setup

Import the helper functions and set the analysis parameters. To switch to another EE_backbone export, change `CONNECTIVITY_PATH`. If a matching netlist is available, set `METADATA_NETLIST_PATH` so E/I labels come from explicit metadata rather than inference.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
    HAS_PLOTS = True
except ImportError:
    HAS_PLOTS = False

import sys
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for candidate in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(candidate) not in sys.path:
        sys.path.append(str(candidate))

from inhibitory_modulation import (
    load_connectivity,
    run_side_by_side_analysis,
    save_side_by_side_outputs,
    source_receiver_block_view,
)

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 40)
pd.set_option("display.precision", 4)

CONNECTIVITY_PATH = PROJECT_ROOT / "matrices" / "mij_matrix.csv"
METADATA_NETLIST_PATH = PROJECT_ROOT / "matrices" / "mij_netlist.csv"
MATRIX_ORIENTATION = "pre_by_post"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "02_inhibitory_modulation_analysis"

TARGET_RHO = 1.0
ALPHA = 0.85
N_NULL = 250
CLASS_MAP = {}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CONNECTIVITY_PATH

## Load, Orient, And Label The Backbone

The loader normalizes the matrix convention to `M[receiver, source]`. When `METADATA_NETLIST_PATH` exists, E/I labels are read from the netlist; otherwise the helper falls back to explicit `CLASS_MAP`, neuron-name hints, and signed outgoing weights. The expected output is the matrix size, E/I population counts, and a small matrix preview.

In [ ]:
data = load_connectivity(
    CONNECTIVITY_PATH,
    metadata_netlist_path=METADATA_NETLIST_PATH,
    class_map=CLASS_MAP,
    matrix_orientation=MATRIX_ORIENTATION,
)

print(f"Loaded matrix: {data.source}")
print(f"Matrix shape: {data.matrix.shape[0]} x {data.matrix.shape[1]}")
display(data.classes.value_counts().rename(index={"e": "excitatory", "i": "inhibitory"}).to_frame("population_count"))
display(data.matrix.iloc[:5, :5])

## Normalize With-Self And No-Self Variants

The newer workflow compares two variants: `with_self`, where diagonal/self connections are retained, and `no_self`, where the diagonal is zeroed before normalization. Each variant is independently normalized to `TARGET_RHO`, so a dominant value of 1.0 is the chosen reference scale rather than a raw measurement.

In [ ]:
results = run_side_by_side_analysis(
    data,
    normalization="spectral_radius",
    target=TARGET_RHO,
    alpha=ALPHA,
    n_null=N_NULL,
)

display(results["normalization"])

## Build EE, EI, IE, And II Blocks

This table summarizes source-to-receiver blocks using the newer convention: `EI` means E source to I receiver, while `IE` means I source to E receiver. The expected output is a side-by-side block table with density, sign counts, and Frobenius norms for both variants.

In [ ]:
block_rows = []
for variant, payload in results["variants"].items():
    for block_name, block in source_receiver_block_view(payload["matrix"], data.classes).items():
        values = block.to_numpy(dtype=float)
        nonzero = values[values != 0]
        block_rows.append({
            "variant": variant,
            "block": block_name,
            "shape": str(block.shape),
            "nonzero": int(np.count_nonzero(values)),
            "positive": int(np.count_nonzero(values > 0)),
            "negative": int(np.count_nonzero(values < 0)),
            "density": np.count_nonzero(values) / values.size if values.size else np.nan,
            "median": np.median(nonzero) if len(nonzero) else 0.0,
            "abs_median": np.median(np.abs(nonzero)) if len(nonzero) else 0.0,
            "fro_norm": np.linalg.norm(values, "fro"),
        })
block_summary = pd.DataFrame(block_rows)
display(block_summary)

## Newer Perturbation Analysis: Remove EE, EI, IE, II, Or Self-Connections

This replaces the earlier scale-only perturbation as the primary perturbation table. Each row removes one source-to-receiver block or the diagonal self-connection group, recomputes the dominant eigenvalue exactly, and compares it with a first-order perturbation estimate from the dominant left/right eigenvectors. If removal increases the dominant real part, the removed group was stabilizing relative to the normalized baseline.

In [ ]:
perturbation = results["perturbation"].sort_values(["removed_block", "variant"])
display(perturbation)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(9, 4), dpi=140)
    sns.barplot(data=perturbation, x="removed_block", y="delta_dominant_real", hue="variant", ax=ax, palette="vlag")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Dominant eigenvalue real-part shift after block removal")
    ax.set_xlabel("Removed source->receiver block")
    ax.set_ylabel("Delta dominant real part")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "block_removal_delta_dominant_real_by_variant.png", bbox_inches="tight")
    plt.show()

## Schur Complement: Eliminate Inhibitory Nodes

The newer Schur analysis uses the eigenvalue-dependent effective E operator `M_eff_E(z) = M_EE + M_EI inv(zI - M_II) M_IE`, where `z` defaults to the dominant eigenvalue of each full variant. The expected output compares direct E dynamics, inhibitory feedback, and the effective E operator.

In [ ]:
schur_summary = results["schur_summary"]
display(schur_summary)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(8, 4), dpi=140)
    sns.barplot(data=schur_summary, x="component", y="dominant_real", hue="variant", ax=ax, palette="Set2")
    ax.axhline(0, color="black", linewidth=1)
    ax.set_title("Schur component dominant real part")
    ax.set_xlabel("")
    ax.set_ylabel("Dominant real part")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "schur_summary_by_variant.png", bbox_inches="tight")
    plt.show()

## Resolvent Response To Exciting E Or I Populations

This section applies the stable discrete resolvent `(I - alpha M)^-1 u` after stimulating all inhibitory or all excitatory populations. Positive E responses after inhibitory excitation are candidate disinhibitory effects, while large I responses point to candidate inhibitory relays.

In [ ]:
print("Top response to exciting all inhibitory populations")
display(results["response_I"])
print("Top response to exciting all excitatory populations")
display(results["response_E"])

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=140)
    for ax, (title, frame) in zip(axes, [("All I excited", results["response_I"]), ("All E excited", results["response_E"])]):
        sns.barplot(data=frame.groupby("variant", group_keys=False).head(10), y="cell", x="response", hue="variant", ax=ax)
        ax.axvline(0, color="black", linewidth=1)
        ax.set_title(title)
        ax.set_xlabel("Resolvent response")
        ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "population_excitation_responses_by_variant.png", bbox_inches="tight")
    plt.show()

## Motif Enrichment And Motif-Removal Stability

The motif table includes broad motifs plus `self_connections` as an explicit group. The companion causal perturbation asks: if edges participating in a motif class, including the diagonal self-connection class, are removed, how does the dominant eigenvalue shift? This version also reports every excitatory population touched by each motif edge set, split into source, receiver, and any-role lists.

In [ ]:
motif = results["motif"]
motif_stability = results["motif_stability"]

affected_e_columns = [
    "variant",
    "motif",
    "affected_excitatory_count",
    "affected_excitatory_cells",
    "affected_excitatory_source_count",
    "affected_excitatory_sources",
    "affected_excitatory_receiver_count",
    "affected_excitatory_receivers",
]

print("Motif enrichment/counts")
display(motif.sort_values(["motif", "variant"]))
print("Excitatory populations affected by motif edge sets")
display(motif_stability.sort_values(["motif", "variant"])[affected_e_columns])
print("Motif-removal dominant eigenvalue stability effects")
display(motif_stability.sort_values(["motif", "variant"]))

if HAS_PLOTS:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=140)
    sns.barplot(data=motif, y="motif", x="z_score", hue="variant", ax=axes[0], palette="crest")
    axes[0].axvline(0, color="black", linewidth=1)
    axes[0].set_title("Motif enrichment against shuffled-weight null")
    axes[0].set_xlabel("Null z-score")
    axes[0].set_ylabel("")
    sns.barplot(data=motif_stability, y="motif", x="delta_dominant_real", hue="variant", ax=axes[1], palette="vlag")
    axes[1].axvline(0, color="black", linewidth=1)
    axes[1].set_title("Motif-removal stability effect")
    axes[1].set_xlabel("Delta dominant real eigenvalue")
    axes[1].set_ylabel("")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "motif_enrichment_and_stability_by_variant.png", bbox_inches="tight")
    plt.show()


## Neumann Series And Chain-Depth Model Selection

The Neumann expansion decomposes inhibitory feedback into `E->I->E`, `E->I->I->E`, and longer chained I-I terms. This merged version keeps the original decay, cumulative energy, and null significance rules, and adds the newer spectral-tail rule based on `alpha * rho(M_II)`. The expected output is a per-order contribution table plus selected order by rule and by variant.

In [ ]:
neumann = results["neumann"]
print("Selected order by variant")
display(neumann["selected_orders"])
print("Selected order by rule")
display(neumann["selected_by_rule"])
display(neumann["selection_table"])

if HAS_PLOTS:
    selection_table = neumann["selection_table"]
    fig, ax = plt.subplots(figsize=(9, 4.5), dpi=140)
    sns.lineplot(data=selection_table, x="ii_chain_order", y="relative_contribution", hue="variant", marker="o", ax=ax)
    sns.lineplot(data=selection_table, x="ii_chain_order", y="spectral_bound", hue="variant", marker="s", linestyle="--", ax=ax, legend=False)
    ax.set_xlabel("I-I chain order")
    ax.set_ylabel("Relative contribution / spectral bound")
    ax.set_title("Neumann model selection for inhibitory-chain depth")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "neumann_ii_model_selection_by_variant.png", bbox_inches="tight")
    plt.show()

## Candidate Disinhibitory Relays

This ranking scores inhibitory populations by their I-to-E strength, E-to-I recruitment, and participation in the selected I-I chain order. The expected output is a shortlist of inhibitory cell classes that may carry stabilizing disinhibitory feedback.

In [ ]:
disinhibitors = results["disinhibitors"]
display(disinhibitors)

if HAS_PLOTS:
    fig, ax = plt.subplots(figsize=(10, 6), dpi=140)
    sns.barplot(data=disinhibitors.groupby("variant", group_keys=False).head(12), y="inhibitory_cell", x="score", hue="variant", ax=ax)
    ax.set_title("Top candidate disinhibitory stabilizers")
    ax.set_xlabel("Composite disinhibition score")
    ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "top_disinhibitory_sources_by_variant.png", bbox_inches="tight")
    plt.show()

## Explicit E-I-E And E-I-I-E Configurations

The path tables list the strongest explicit inhibitory-feedback and disinhibitory-feedback configurations. `E-I-E` is a two-hop inhibitory route; `E-I-I-E` is the shortest disinhibitory route, where two inhibitory edges create a positive E-to-E contribution under expected signs.

In [ ]:
paths = results["paths"]
print("Top E-I-E configurations")
display(paths["EIE"])
print("Top E-I-I-E disinhibitory configurations")
display(paths["EIIE"])

if HAS_PLOTS and not paths["EIE"].empty and not paths["EIIE"].empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=140)
    for ax, title, frame, xcol in [
        (axes[0], "Top E-I-E", paths["EIE"].groupby("variant", group_keys=False).head(10), "abs_contribution"),
        (axes[1], "Top E-I-I-E", paths["EIIE"].groupby("variant", group_keys=False).head(10), "signed_contribution"),
    ]:
        plot_frame = frame.copy()
        plot_frame["configuration"] = plot_frame.apply(
            lambda r: f"{r['E_source']} -> {r.get('I_middle', r.get('I_first'))} -> {r.get('I_second', '')} -> {r['E_receiver']}".replace(" ->  -> ", " -> "),
            axis=1,
        )
        sns.barplot(data=plot_frame, y="configuration", x=xcol, hue="variant", ax=ax)
        ax.set_title(title)
        ax.set_ylabel("")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "top_path_configurations_by_variant.png", bbox_inches="tight")
    plt.show()

## Takeaways

- Use this notebook after the backbone workflow: it assumes the signed matrix, E/I labels, orientation, normalization, and self-connection handling are already conceptually defined.
- Keep the `with_self` and `no_self` results side by side throughout; the with-self variant is the primary self-inclusive analysis, while the no-self variant remains a sensitivity contrast.
- Self-connections are now tested directly in the block-removal table and in motif enrichment/removal stability as the `SELF` / `self_connections` group.
- Use the consolidated triad sensitivity notebook as a local motif companion rather than as a prerequisite for this global E/I analysis.

## Save Tables

The final step saves the side-by-side tables to `outputs/02_inhibitory_modulation_analysis`. Expected outputs include perturbation, Schur, response, motif, Neumann, disinhibitory-source, and explicit-path CSVs.

In [ ]:
save_side_by_side_outputs(OUTPUT_DIR, results)
print(f"Saved side-by-side tables and figures to {OUTPUT_DIR.resolve()}")